# EgoBrain — Visual Feature Extraction Worker

**Purpose:** Extract frozen 512-D one-second egocentric video embeddings using the R3D-18 Kinetics-400 backbone. The notebook is designed as a parallel Kaggle worker and writes per-subject tensors.

## Reproducibility contract

Run this notebook independently for each worker ID. The production extraction used five workers covering the 40 subjects. Do not commit generated feature tensors to Git; publish them as a dataset/input artifact.

### Inputs

- Gated `ut-vision/EgoBrain` dataset access.
- Kaggle secret `HF_TOKEN`.
- CUDA GPU (the original worker is designed for GPU execution).

### Outputs

- `/kaggle/working/features/features_Pxxxx.pt`
- `/kaggle/working/egobrain_worker_<id>_features.zip`

---



In [ ]:
# ============================================================
# EgoBrain Visual Feature Extraction Worker
# ============================================================
#
# Target:
#   Kaggle Dual T4
#   Headless "Save & Run All / Commit"
#
# Dataset:
#   ut-vision/EgoBrain
#
# Output:
#   /kaggle/working/features/features_Pxxxx.pt
#
# Each subject tensor:
#   [Num_1sec_Clips, 512]
#   dtype = torch.float16
#
# Backbone:
#   torchvision r3d_18
#   pretrained Kinetics-400 weights
#
# IMPORTANT:
#   r3d_18 officially uses 16-frame clips and 112x112 crops.
#   We therefore default to 112x112 to preserve the published
#   preprocessing and keep the runtime realistic on T4 GPUs.
#
# ============================================================

import os
import sys
import gc
import json
import time
import shutil
import hashlib
import zipfile
import subprocess
from pathlib import Path
from typing import List, Dict, Generator, Tuple

import numpy as np


# ============================================================
# 0. WORKER CONFIGURATION
# ============================================================

WORKER_ID = 4
TOTAL_WORKERS = 5

# Dataset
HF_REPO_ID = "ut-vision/EgoBrain"
HF_REPO_TYPE = "dataset"

# Subject configuration
TOTAL_SUBJECTS = 40
SUBJECT_PREFIX = "P"

# Video layout requested by the dataset
VIDEO_SUBDIR = "GoPro_HERO12_Ego"

# ============================================================
# Temporal sampling
# ============================================================

# 16 frames = approximately 1 second at 16 FPS.
CLIP_FRAMES = 16
TARGET_SAMPLE_FPS = 16

# ============================================================
# Spatial preprocessing
# ============================================================

# r3d_18 official weights use 112x112 crop.
# Changing this to 224 is possible, but substantially increases
# inference cost and deviates from the published preprocessing.
MODEL_INPUT_SIZE = 112

# ============================================================
# Inference
# ============================================================

# Number of 16-frame clips sent through the model together.
#
# Dual T4:
#   DataParallel splits this batch between both GPUs.
#
# 8 = normally safe.
# Increase only after measuring VRAM.
BATCH_CLIPS = 8

# Expected r3d_18 pooled feature dimension.
FEATURE_DIM = 512

# Safety ceiling for worker feature output.
#
# Requirement says <10 GB.
# We use 9 GB to leave headroom.
MAX_FEATURE_OUTPUT_GB = 9.0

# Minimum desired working-disk free space.
MIN_FREE_GB = 3.0

# Hugging Face download concurrency.
#
# Intentionally kept at 1 because each file may be huge.
HF_DOWNLOAD_WORKERS = 1


# ============================================================
# 1. PATHS
# ============================================================

WORKING_DIR = Path("/kaggle/working")
TEMP_DIR = Path("/kaggle/temp")

FEATURE_DIR = WORKING_DIR / "features"
RAW_DOWNLOAD_ROOT = TEMP_DIR / "raw_download"

HF_HOME = TEMP_DIR / "hf_home"
TORCH_HOME = TEMP_DIR / "torch_home"

ERROR_LOG = WORKING_DIR / f"worker_{WORKER_ID}_errors.jsonl"
RUN_MANIFEST = WORKING_DIR / f"worker_{WORKER_ID}_manifest.json"
CHECKSUM_LOG = WORKING_DIR / f"worker_{WORKER_ID}_md5.txt"

ZIP_PATH = WORKING_DIR / f"egobrain_worker_{WORKER_ID}_features.zip"


FEATURE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
RAW_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
TORCH_HOME.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. ENVIRONMENT CONTROL
# ============================================================

# Prevent Hugging Face from using the default:
# ~/.cache/huggingface/
#
# Everything is redirected into temporary scratch storage.
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_XET_CACHE"] = str(HF_HOME / "xet")
os.environ["HF_ASSETS_CACHE"] = str(HF_HOME / "assets")

os.environ["TORCH_HOME"] = str(TORCH_HOME)

# Optional noise reduction.
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# OpenCV/FFmpeg/PyTorch thread configuration can be tuned later.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")


# ============================================================
# 3. VALIDATE WORKER CONFIGURATION
# ============================================================

if TOTAL_SUBJECTS % TOTAL_WORKERS != 0:
    raise ValueError(
        "TOTAL_SUBJECTS must be divisible by TOTAL_WORKERS."
    )

if not (1 <= WORKER_ID <= TOTAL_WORKERS):
    raise ValueError(
        f"WORKER_ID must be between 1 and {TOTAL_WORKERS}."
    )

SUBJECTS_PER_WORKER = TOTAL_SUBJECTS // TOTAL_WORKERS

worker_start_idx = (WORKER_ID - 1) * SUBJECTS_PER_WORKER + 1
worker_end_idx = WORKER_ID * SUBJECTS_PER_WORKER

ASSIGNED_SUBJECTS = [
    f"{SUBJECT_PREFIX}{i:04d}"
    for i in range(worker_start_idx, worker_end_idx + 1)
]


# ============================================================
# 4. BASIC UTILITIES
# ============================================================

def bytes_to_gb(n: int) -> float:
    return n / (1024 ** 3)


def directory_size(path: Path) -> int:
    """Return total recursive file size in bytes."""
    if not path.exists():
        return 0

    total = 0

    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass

    return total


def disk_report():
    """Print working and scratch disk state."""
    print("\n" + "=" * 70)
    print("DISK STATUS")
    print("=" * 70)

    for path in [WORKING_DIR, TEMP_DIR]:
        try:
            usage = shutil.disk_usage(path)

            print(
                f"{path} | "
                f"total={bytes_to_gb(usage.total):.2f} GB | "
                f"used={bytes_to_gb(usage.used):.2f} GB | "
                f"free={bytes_to_gb(usage.free):.2f} GB"
            )
        except Exception as exc:
            print(f"Could not inspect {path}: {exc}")

    feature_bytes = directory_size(FEATURE_DIR)

    print(
        f"Feature directory: "
        f"{bytes_to_gb(feature_bytes):.4f} GB"
    )

    print("=" * 70 + "\n")


def ensure_free_space():
    """Abort before a new download if working disk is already dangerously full."""
    usage = shutil.disk_usage(WORKING_DIR)

    free_gb = bytes_to_gb(usage.free)

    if free_gb < MIN_FREE_GB:
        raise RuntimeError(
            f"Only {free_gb:.2f} GB free in /kaggle/working."
        )


def cleanup_directory(path: Path):
    """Aggressively remove a directory."""
    try:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
    except Exception as exc:
        print(f"[WARN] Could not remove {path}: {exc}")


def cleanup_huggingface_caches():
    """
    Remove all HF cache locations we explicitly control.
    Also remove the default user HF cache as a safety net.
    """

    candidate_paths = [
        HF_HOME,
        Path.home() / ".cache" / "huggingface",
    ]

    for path in candidate_paths:
        cleanup_directory(path)

    HF_HOME.mkdir(parents=True, exist_ok=True)


def memory_cleanup():
    """Release Python and CUDA allocations."""
    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    except Exception:
        pass


def log_error(
    subject_id: str,
    video_path: str,
    error: Exception,
):
    """Append structured error information."""
    record = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "worker_id": WORKER_ID,
        "subject_id": subject_id,
        "video": video_path,
        "error_type": type(error).__name__,
        "error": str(error),
    }

    with ERROR_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


def md5_file(path: Path, chunk_size: int = 16 * 1024 * 1024) -> str:
    """Calculate MD5 without loading the entire file into RAM."""
    md5 = hashlib.md5()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


# ============================================================
# 5. REQUIRED PACKAGES
# ============================================================

try:
    import torch
    import torch.nn as nn
    import torchvision

    from torchvision.models.video import (
        r3d_18,
        R3D_18_Weights,
    )

except ImportError as exc:
    raise RuntimeError(
        "PyTorch/torchvision is missing from the Kaggle environment."
    ) from exc


try:
    from huggingface_hub import (
        snapshot_download,
        login,
    )

except ImportError as exc:
    raise RuntimeError(
        "huggingface_hub is missing. "
        "Install it in a separate Kaggle cell if necessary."
    ) from exc


# ============================================================
# 6. GET HUGGING FACE SECRET
# ============================================================

print("=" * 70)
print("AUTHENTICATING WITH HUGGING FACE")
print("=" * 70)

try:
    from kaggle_secrets import UserSecretsClient

    secrets_client = UserSecretsClient()

    hf_token = secrets_client.get_secret("HF_TOKEN")

except Exception as exc:
    raise RuntimeError(
        "Could not read Kaggle Secret 'HF_TOKEN'."
    ) from exc


if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty."
    )


# Authenticate once.
#
# snapshot_download() also receives token=hf_token explicitly,
# so destroying the on-disk HF home after a subject does not
# prevent later subjects from authenticating.
login(
    token=hf_token,
    add_to_git_credential=False,
)


print("Hugging Face authentication completed.")


# ============================================================
# 7. DEVICE / GPU SETUP
# ============================================================

print("\n" + "=" * 70)
print("GPU CONFIGURATION")
print("=" * 70)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. This worker is designed for GPU execution."
    )

GPU_COUNT = torch.cuda.device_count()

print(f"CUDA devices detected: {GPU_COUNT}")

for gpu_idx in range(GPU_COUNT):
    print(
        f"GPU {gpu_idx}: "
        f"{torch.cuda.get_device_name(gpu_idx)}"
    )

if GPU_COUNT >= 2:
    print("Dual-GPU inference enabled with DataParallel.")
else:
    print("Only one GPU detected; continuing with single-GPU inference.")

torch.backends.cudnn.benchmark = True


# ============================================================
# 8. LOAD FROZEN R3D-18 BACKBONE
# ============================================================

print("\n" + "=" * 70)
print("LOADING R3D-18 BACKBONE")
print("=" * 70)

weights = R3D_18_Weights.DEFAULT

base_model = r3d_18(weights=weights)

# Remove classification head.
#
# Original:
#   ...
#   avgpool
#   fc
#
# We retain avgpool and remove fc.
backbone = nn.Sequential(
    *list(base_model.children())[:-1]
)

backbone.eval()

# Freeze everything.
for parameter in backbone.parameters():
    parameter.requires_grad_(False)


# Move to first CUDA device.
backbone = backbone.cuda()


# Use both T4s when available.
if GPU_COUNT >= 2:
    backbone = nn.DataParallel(backbone)


print(
    f"Backbone: torchvision r3d_18 "
    f"({FEATURE_DIM}-D pooled features)"
)

print(f"Input clip: {CLIP_FRAMES} frames")
print(f"Sampling rate: {TARGET_SAMPLE_FPS} FPS")
print(f"Spatial input: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
print(f"Batch clips: {BATCH_CLIPS}")


# ============================================================
# 9. KINETICS NORMALIZATION
# ============================================================

# These are the published normalization values associated with
# R3D-18 Kinetics-400 pretrained weights.
#
# Do NOT replace these with ImageNet normalization unless you
# deliberately change the feature extractor.

KINETICS_MEAN = torch.tensor(
    [0.43216, 0.394666, 0.37645],
    dtype=torch.float32,
)

KINETICS_STD = torch.tensor(
    [0.22803, 0.22145, 0.216989],
    dtype=torch.float32,
)


# GPU copies.
KINETICS_MEAN_GPU = (
    KINETICS_MEAN
    .view(1, 1, 1, 1, 3)
    .cuda()
)

KINETICS_STD_GPU = (
    KINETICS_STD
    .view(1, 1, 1, 1, 3)
    .cuda()
)


# ============================================================
# 10. FFmpeg VALIDATION
# ============================================================

FFMPEG_BIN = shutil.which("ffmpeg")

if FFMPEG_BIN is None:
    raise RuntimeError(
        "FFmpeg was not found in the Kaggle environment."
    )

print(f"FFmpeg: {FFMPEG_BIN}")


# ============================================================
# 11. FRAME STREAMER
# ============================================================

def stream_video_clips(
    video_path: Path,
) -> Generator[np.ndarray, None, None]:
    """
    Decode one video through FFmpeg.

    Pipeline inside FFmpeg:

        4K video
            ↓
        temporal resampling @ 16 FPS
            ↓
        spatial resize
            ↓
        RGB frames
            ↓
        Python receives only 112x112 frames

    Each yielded array has shape:

        [16, H, W, 3]

    dtype:
        uint8

    Memory remains bounded.
    """

    vf = (
        f"fps={TARGET_SAMPLE_FPS},"
        f"scale={MODEL_INPUT_SIZE}:{MODEL_INPUT_SIZE}:flags=area"
    )

    command = [
        FFMPEG_BIN,

        "-hide_banner",
        "-loglevel", "error",

        "-i", str(video_path),

        "-vf", vf,

        "-pix_fmt", "rgb24",

        "-f", "rawvideo",

        "pipe:1",
    ]

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1024 * 1024,
    )

    frame_bytes = (
        MODEL_INPUT_SIZE
        * MODEL_INPUT_SIZE
        * 3
    )

    clip_frames = []

    try:

        while True:

            raw = process.stdout.read(frame_bytes)

            if not raw:
                break

            # Ignore malformed partial frames.
            if len(raw) != frame_bytes:
                break

            frame = np.frombuffer(
                raw,
                dtype=np.uint8,
            ).reshape(
                MODEL_INPUT_SIZE,
                MODEL_INPUT_SIZE,
                3,
            )

            clip_frames.append(frame)

            if len(clip_frames) == CLIP_FRAMES:

                clip = np.stack(
                    clip_frames,
                    axis=0,
                )

                yield clip

                clip_frames.clear()

    finally:

        if process.stdout is not None:
            process.stdout.close()

        stderr_bytes = b""

        try:
            stderr_bytes = process.stderr.read()
        except Exception:
            pass

        return_code = process.wait()

        if return_code != 0:

            stderr_text = (
                stderr_bytes.decode(
                    "utf-8",
                    errors="replace",
                ).strip()
            )

            raise RuntimeError(
                f"FFmpeg failed with exit code "
                f"{return_code}: {stderr_text[-2000:]}"
            )

        # Deliberately discard incomplete final clips.
        clip_frames.clear()


# ============================================================
# 12. PREPARE CLIP BATCH FOR R3D
# ============================================================

def prepare_batch(
    clips: List[np.ndarray],
) -> torch.Tensor:
    """
    Convert:

        List[np.ndarray]
        each [T,H,W,C] uint8

    into:

        [B,C,T,H,W] float16
    """

    # [B,T,H,W,C]
    batch_np = np.stack(
        clips,
        axis=0,
    )

    # Avoid unnecessary float64 conversion.
    batch = torch.from_numpy(batch_np)

    # uint8 -> float32 -> [0,1]
    batch = batch.float().div_(255.0)

    # Normalize in RGB space.
    batch = (
        batch
        - KINETICS_MEAN_GPU.cpu()
    ) / KINETICS_STD_GPU.cpu()

    # [B,T,H,W,C] -> [B,C,T,H,W]
    batch = batch.permute(
        0, 4, 1, 2, 3
    )

    # Move directly to GPU and use fp16.
    batch = batch.pin_memory().cuda(
        non_blocking=True
    )

    batch = batch.half()

    return batch


# ============================================================
# 13. MODEL INFERENCE
# ============================================================

@torch.inference_mode()
def extract_embeddings(
    clips: List[np.ndarray],
) -> torch.Tensor:
    """
    Input:
        list of [16,H,W,3] uint8 clips

    Output:
        [B,512] float16 CPU tensor
    """

    batch = prepare_batch(clips)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):

        features = backbone(batch)

    # Expected:
    #
    # [B,512,1,1,1]
    #
    # Flatten pooled spatial/temporal dimensions.
    features = features.flatten(
        start_dim=1
    )

    if features.ndim != 2:
        raise RuntimeError(
            f"Unexpected feature shape: "
            f"{tuple(features.shape)}"
        )

    if features.shape[1] != FEATURE_DIM:
        raise RuntimeError(
            f"Unexpected feature dimension: "
            f"{features.shape[1]} "
            f"(expected {FEATURE_DIM})"
        )

    # Bring embeddings back to CPU.
    features = features.detach().cpu().half()

    return features


# ============================================================
# 14. EXTRACT ONE VIDEO
# ============================================================

def process_video(
    subject_id: str,
    video_path: Path,
) -> Tuple[torch.Tensor, int]:

    print(
        f"\n[{subject_id}] "
        f"Processing video: {video_path.name}"
    )

    video_start = time.time()

    embeddings = []

    clip_batch = []

    total_clips = 0

    try:

        for clip in stream_video_clips(video_path):

            clip_batch.append(clip)

            if len(clip_batch) >= BATCH_CLIPS:

                features = extract_embeddings(
                    clip_batch
                )

                embeddings.append(features)

                total_clips += features.shape[0]

                clip_batch.clear()

        # Process final incomplete inference batch.
        if clip_batch:

            features = extract_embeddings(
                clip_batch
            )

            embeddings.append(features)

            total_clips += features.shape[0]

            clip_batch.clear()

        if embeddings:

            result = torch.cat(
                embeddings,
                dim=0,
            )

        else:

            result = torch.empty(
                (0, FEATURE_DIM),
                dtype=torch.float16,
            )

        elapsed = time.time() - video_start

        print(
            f"[{subject_id}] "
            f"Completed {video_path.name} | "
            f"clips={total_clips:,} | "
            f"time={elapsed / 60:.2f} min"
        )

        return result, total_clips

    finally:

        # Explicitly satisfy the per-video cleanup requirement.
        embeddings.clear()
        clip_batch.clear()

        memory_cleanup()


# ============================================================
# 15. DOWNLOAD ONE SUBJECT
# ============================================================

def download_subject(
    subject_id: str,
) -> Path:

    cleanup_directory(RAW_DOWNLOAD_ROOT)
    RAW_DOWNLOAD_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    ensure_free_space()

    print("\n" + "=" * 70)
    print(
        f"DOWNLOADING {subject_id}"
    )
    print("=" * 70)

    allow_patterns = [
        f"{subject_id}/{VIDEO_SUBDIR}/*.MP4",
        f"{subject_id}/{VIDEO_SUBDIR}/*.mp4",
    ]

    local_snapshot = snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,

        allow_patterns=allow_patterns,

        # Critical:
        # Download into our dedicated scratch directory rather
        # than the normal ~/.cache/huggingface/hub/.
        local_dir=str(RAW_DOWNLOAD_ROOT),

        # Keep download concurrency conservative.
        max_workers=HF_DOWNLOAD_WORKERS,

        # Explicit auth token.
        token=hf_token,

        # Force fresh subject-local retrieval.
        force_download=False,
    )

    return Path(local_snapshot)


# ============================================================
# 16. FIND SUBJECT MP4 FILES
# ============================================================

def find_subject_videos(
    subject_id: str,
) -> List[Path]:

    subject_dir = RAW_DOWNLOAD_ROOT / subject_id

    if not subject_dir.exists():

        # In case snapshot_download returned a slightly
        # different path structure, search recursively.
        candidates = [
            p
            for p in RAW_DOWNLOAD_ROOT.rglob("*")
            if p.is_file()
            and p.suffix.lower() == ".mp4"
            and subject_id in p.parts
        ]

    else:

        candidates = [
            p
            for p in subject_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() == ".mp4"
        ]

    candidates = sorted(
        candidates,
        key=lambda p: str(p),
    )

    return candidates


# ============================================================
# 17. PROCESS ONE SUBJECT
# ============================================================

def process_subject(
    subject_id: str,
) -> Dict:

    output_path = (
        FEATURE_DIR
        / f"features_{subject_id}.pt"
    )

    # --------------------------------------------------------
    # Idempotency checkpoint
    # --------------------------------------------------------

    if output_path.exists():

        print(
            f"\n[{subject_id}] "
            f"Checkpoint exists -> SKIPPING"
        )

        tensor = torch.load(
            output_path,
            map_location="cpu",
            weights_only=True,
        )

        return {
            "subject_id": subject_id,
            "status": "skipped_existing",
            "num_clips": int(tensor.shape[0]),
            "feature_dim": int(tensor.shape[1]),
            "bytes": output_path.stat().st_size,
        }

    subject_start = time.time()

    all_subject_embeddings = []

    successful_videos = 0
    failed_videos = 0

    video_records = []

    try:

        # ----------------------------------------------------
        # Download
        # ----------------------------------------------------

        download_subject(subject_id)

        # ----------------------------------------------------
        # Discover videos
        # ----------------------------------------------------

        video_files = find_subject_videos(
            subject_id
        )

        print(
            f"[{subject_id}] "
            f"Found {len(video_files)} MP4 files."
        )

        if not video_files:

            raise RuntimeError(
                f"No MP4 files found for {subject_id}."
            )

        # ----------------------------------------------------
        # Extract video by video
        # ----------------------------------------------------

        for video_number, video_path in enumerate(
            video_files,
            start=1,
        ):

            print(
                f"\n[{subject_id}] "
                f"VIDEO {video_number}/{len(video_files)}"
            )

            try:

                features, clip_count = process_video(
                    subject_id,
                    video_path,
                )

                if features.numel() > 0:

                    all_subject_embeddings.append(
                        features
                    )

                successful_videos += 1

                video_records.append({
                    "video": str(video_path),
                    "status": "success",
                    "num_clips": int(clip_count),
                })

            except Exception as exc:

                failed_videos += 1

                print(
                    f"[ERROR] "
                    f"{subject_id} / "
                    f"{video_path.name}: "
                    f"{exc}"
                )

                log_error(
                    subject_id,
                    str(video_path),
                    exc,
                )

                video_records.append({
                    "video": str(video_path),
                    "status": "failed",
                    "error": str(exc),
                })

                # IMPORTANT:
                # Do NOT crash the entire subject because
                # one video was corrupted.
                continue

            finally:

                # Explicit per-video resource cleanup.
                memory_cleanup()

        # ----------------------------------------------------
        # Consolidate subject embeddings
        # ----------------------------------------------------

        if all_subject_embeddings:

            subject_features = torch.cat(
                all_subject_embeddings,
                dim=0,
            ).half()

        else:

            subject_features = torch.empty(
                (0, FEATURE_DIM),
                dtype=torch.float16,
            )

        # ----------------------------------------------------
        # Validate tensor
        # ----------------------------------------------------

        if subject_features.ndim != 2:
            raise RuntimeError(
                f"{subject_id}: invalid tensor ndim "
                f"{subject_features.ndim}"
            )

        if subject_features.shape[1] != FEATURE_DIM:
            raise RuntimeError(
                f"{subject_id}: feature dimension "
                f"{subject_features.shape[1]} "
                f"!= {FEATURE_DIM}"
            )

        expected_bytes = (
            subject_features.numel()
            * subject_features.element_size()
        )

        # ----------------------------------------------------
        # Global output ceiling
        # ----------------------------------------------------

        existing_feature_bytes = directory_size(
            FEATURE_DIR
        )

        projected_total = (
            existing_feature_bytes
            + expected_bytes
        )

        projected_gb = bytes_to_gb(
            projected_total
        )

        if projected_gb > MAX_FEATURE_OUTPUT_GB:

            raise RuntimeError(
                f"Feature output ceiling exceeded. "
                f"Projected={projected_gb:.3f} GB "
                f"Limit={MAX_FEATURE_OUTPUT_GB:.3f} GB"
            )

        # ----------------------------------------------------
        # Save atomically
        # ----------------------------------------------------

        temp_output = output_path.with_suffix(
            ".pt.tmp"
        )

        if temp_output.exists():
            temp_output.unlink()

        torch.save(
            subject_features,
            temp_output,
        )

        temp_output.replace(
            output_path
        )

        # ----------------------------------------------------
        # Subject report
        # ----------------------------------------------------

        elapsed = time.time() - subject_start

        status = (
            "success"
            if failed_videos == 0
            else "partial_video_failures"
        )

        record = {
            "subject_id": subject_id,
            "status": status,
            "num_videos": len(video_files),
            "successful_videos": successful_videos,
            "failed_videos": failed_videos,
            "num_clips": int(subject_features.shape[0]),
            "feature_dim": int(subject_features.shape[1]),
            "dtype": str(subject_features.dtype),
            "feature_bytes": int(output_path.stat().st_size),
            "elapsed_seconds": elapsed,
            "videos": video_records,
        }

        print("\n" + "-" * 70)
        print(
            f"[{subject_id}] SUBJECT COMPLETE"
        )
        print(
            f"Clips: {subject_features.shape[0]:,}"
        )
        print(
            f"Feature dim: {subject_features.shape[1]}"
        )
        print(
            f"Output size: "
            f"{bytes_to_gb(output_path.stat().st_size):.4f} GB"
        )
        print(
            f"Failed videos: {failed_videos}"
        )
        print(
            f"Time: {elapsed / 60:.2f} min"
        )
        print("-" * 70)

        return record

    finally:

        # Release subject embedding references.
        all_subject_embeddings.clear()

        memory_cleanup()

        # ----------------------------------------------------
        # CRITICAL SANITATION
        # ----------------------------------------------------

        cleanup_directory(
            RAW_DOWNLOAD_ROOT
        )

        cleanup_huggingface_caches()

        memory_cleanup()

        disk_report()


# ============================================================
# 18. MAIN WORKER LOOP
# ============================================================

def main():

    print("\n" + "=" * 80)
    print("EgoBrain Visual Feature Extraction")
    print("=" * 80)

    print(f"Worker ID: {WORKER_ID}")
    print(f"Total workers: {TOTAL_WORKERS}")

    print(
        f"Assigned subjects: "
        f"{ASSIGNED_SUBJECTS[0]} -> "
        f"{ASSIGNED_SUBJECTS[-1]}"
    )

    print(
        f"Model input: "
        f"{CLIP_FRAMES} frames @ "
        f"{TARGET_SAMPLE_FPS} FPS"
    )

    print(
        f"Spatial resolution: "
        f"{MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}"
    )

    print(
        f"Output feature dimension: "
        f"{FEATURE_DIM}"
    )

    disk_report()

    worker_start = time.time()

    records = []
    subject_failures = []

    # --------------------------------------------------------
    # Subject loop
    # --------------------------------------------------------

    for subject_number, subject_id in enumerate(
        ASSIGNED_SUBJECTS,
        start=1,
    ):

        print("\n")
        print("#" * 80)
        print(
            f"WORKER {WORKER_ID} | "
            f"SUBJECT {subject_number}/"
            f"{len(ASSIGNED_SUBJECTS)}: "
            f"{subject_id}"
        )
        print("#" * 80)

        try:

            result = process_subject(
                subject_id
            )

            records.append(result)

        except Exception as exc:

            print(
                f"\n[FATAL SUBJECT ERROR] "
                f"{subject_id}: {exc}"
            )

            subject_failures.append({
                "subject_id": subject_id,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })

            log_error(
                subject_id,
                "<SUBJECT LEVEL FAILURE>",
                exc,
            )

            # Continue with next subject.
            continue

    # --------------------------------------------------------
    # Save worker manifest
    # --------------------------------------------------------

    worker_elapsed = (
        time.time()
        - worker_start
    )

    manifest = {
        "worker_id": WORKER_ID,
        "total_workers": TOTAL_WORKERS,
        "assigned_subjects": ASSIGNED_SUBJECTS,
        "records": records,
        "subject_failures": subject_failures,
        "feature_dimension": FEATURE_DIM,
        "clip_frames": CLIP_FRAMES,
        "sample_fps": TARGET_SAMPLE_FPS,
        "model_input_size": MODEL_INPUT_SIZE,
        "backbone": "torchvision.r3d_18",
        "dtype": "torch.float16",
        "elapsed_seconds": worker_elapsed,
    }

    RUN_MANIFEST.write_text(
        json.dumps(
            manifest,
            indent=2,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Final feature size
    # --------------------------------------------------------

    feature_bytes = directory_size(
        FEATURE_DIR
    )

    print("\n" + "=" * 80)
    print("WORKER EXTRACTION FINISHED")
    print("=" * 80)

    print(
        f"Feature directory size: "
        f"{bytes_to_gb(feature_bytes):.4f} GB"
    )

    print(
        f"Worker elapsed time: "
        f"{worker_elapsed / 3600:.2f} hours"
    )

    print(
        f"Completed subjects: "
        f"{len(records)}/{len(ASSIGNED_SUBJECTS)}"
    )

    print(
        f"Subject-level failures: "
        f"{len(subject_failures)}"
    )

    # --------------------------------------------------------
    # Final MD5 generation
    # --------------------------------------------------------

    print("\nGenerating MD5 checksums...")

    feature_files = sorted(
        FEATURE_DIR.glob("features_*.pt")
    )

    with CHECKSUM_LOG.open(
        "w",
        encoding="utf-8",
    ) as f:

        for path in feature_files:

            checksum = md5_file(path)

            f.write(
                f"{checksum}  {path.name}\n"
            )

            print(
                f"{path.name}: {checksum}"
            )

    # --------------------------------------------------------
    # ZIP PACKAGE
    # --------------------------------------------------------

    print("\nCreating feature ZIP package...")

    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    # STORED is intentional:
    #
    #   - fp16 tensors are already compact
    #   - DEFLATE can cost significant CPU time
    #   - this is primarily a bundling operation
    #
    with zipfile.ZipFile(
        ZIP_PATH,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:

        for path in feature_files:

            archive.write(
                path,
                arcname=path.name,
            )

    # Add package checksum.
    zip_md5 = md5_file(
        ZIP_PATH
    )

    with CHECKSUM_LOG.open(
        "a",
        encoding="utf-8",
    ) as f:

        f.write(
            f"{zip_md5}  {ZIP_PATH.name}\n"
        )

    # --------------------------------------------------------
    # Final size verification
    # --------------------------------------------------------

    working_size = directory_size(
        WORKING_DIR
    )

    working_size_gb = bytes_to_gb(
        working_size
    )

    print("\n" + "=" * 80)
    print("FINAL PACKAGE")
    print("=" * 80)

    print(
        f"Feature files: "
        f"{len(feature_files)}"
    )

    print(
        f"Features total: "
        f"{bytes_to_gb(feature_bytes):.4f} GB"
    )

    print(
        f"ZIP size: "
        f"{bytes_to_gb(ZIP_PATH.stat().st_size):.4f} GB"
    )

    print(
        f"Total /kaggle/working usage: "
        f"{working_size_gb:.4f} GB"
    )

    print(
        f"ZIP MD5: "
        f"{zip_md5}"
    )

    print(
        f"Manifest: "
        f"{RUN_MANIFEST}"
    )

    print(
        f"Checksums: "
        f"{CHECKSUM_LOG}"
    )

    disk_report()

    # --------------------------------------------------------
    # Final correctness checks
    # --------------------------------------------------------

    missing_subjects = []

    for subject_id in ASSIGNED_SUBJECTS:

        expected = (
            FEATURE_DIR
            / f"features_{subject_id}.pt"
        )

        if not expected.exists():
            missing_subjects.append(
                subject_id
            )

    if missing_subjects:

        print(
            "\nWARNING: Missing subject outputs:"
        )

        print(
            missing_subjects
        )

    if subject_failures:

        print(
            "\nWARNING: Subject-level failures occurred:"
        )

        for failure in subject_failures:
            print(
                failure
            )

        print(
            "\nWorker did not silently hide the failures. "
            "Successful feature files remain valid and "
            "can be reused on the next run."
        )

    print("\n" + "=" * 80)
    print("DONE")
    print("=" * 80)


# ============================================================
# 19. EXECUTE
# ============================================================

if __name__ == "__main__":
    main()